# StrideZone — Análise de Dados (exemplo didático)

**Importante:** a planilha fornecida é fictícia e serve apenas para testar o processo. Os resultados não representam o desempenho real da StrideZone.

## Como usar
1. Baixe `StrideZone_Planilha_Ficticia.xlsx` para seu computador.
2. Execute as células na ordem, usando o botão ▶ ou `Shift + Enter`.
3. Quando a janela de upload aparecer, selecione a planilha baixada.
4. O notebook vai mostrar os KPIs, estatísticas, correlações, gráficos e exportar arquivos CSV.


## Bibliotecas utilizadas (documentação obrigatória)

| Biblioteca | Para que é usada neste notebook |
|---|---|
| `pandas` | Leitura da planilha, limpeza (ETL), agrupamentos e cálculo dos KPIs |
| `numpy` | Operações numéricas (mínimos, valores ausentes) |
| `matplotlib` | Todos os gráficos |
| `scipy.stats` | Correlações de Pearson e Spearman |
| `google.colab` | Upload da planilha e download dos CSVs (funciona apenas no Google Colab) |
| `openpyxl` | Usada pelo `pandas` para ler arquivos `.xlsx` (já vem instalada no Colab) |

In [ ]:
# 1. Bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from google.colab import files

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [ ]:
# 2. Enviar a planilha fictícia (ou futuramente a planilha do grupo)
uploaded = files.upload()
arquivo = next(iter(uploaded))

# A planilha deve conter as abas "Vendas" e "Estoque"
df_vendas_original = pd.read_excel(arquivo, sheet_name="Vendas")
df_estoque_original = pd.read_excel(arquivo, sheet_name="Estoque")

print("Arquivo carregado:", arquivo)
print("Linhas na aba Vendas:", len(df_vendas_original))
print("Linhas na aba Estoque:", len(df_estoque_original))

display(df_vendas_original.head())
display(df_estoque_original.head())


In [ ]:
# 3. ETL — limpeza e preparação dos dados
df_vendas = df_vendas_original.copy()
df_estoque = df_estoque_original.copy()

# Padronizar datas
df_vendas["data"] = pd.to_datetime(df_vendas["data"], errors="coerce")
df_estoque["mes"] = pd.to_datetime(df_estoque["mes"], errors="coerce")

# Corrigir tipos numéricos
colunas_numericas = [
    "quantidade", "preco_unitario", "custo_unitario",
    "desconto_pct", "unidades_devolvidas"
]
for coluna in colunas_numericas:
    df_vendas[coluna] = pd.to_numeric(df_vendas[coluna], errors="coerce")

for coluna in ["dias_observados", "dias_indisponivel"]:
    df_estoque[coluna] = pd.to_numeric(df_estoque[coluna], errors="coerce")

duplicatas = int(df_vendas.duplicated().sum())
df_vendas = df_vendas.drop_duplicates()

# Valores ausentes: preencher apenas campos em que há regra razoável.
df_vendas["cor"] = df_vendas["cor"].fillna("Não informado")
df_vendas["desconto_pct"] = df_vendas["desconto_pct"].fillna(0)

# Remover registros sem campos essenciais ou com valores inválidos
df_vendas = df_vendas.dropna(
    subset=["pedido_id", "data", "modelo", "linha", "quantidade",
            "preco_unitario", "custo_unitario", "unidades_devolvidas"]
)
df_vendas = df_vendas[
    (df_vendas["quantidade"] > 0) &
    (df_vendas["preco_unitario"] >= 0) &
    (df_vendas["custo_unitario"] >= 0)
].copy()

# Desconto deve estar entre 0 e 1 (por exemplo, 0.10 = 10%)
df_vendas["desconto_pct"] = df_vendas["desconto_pct"].clip(0, 1)

# Devoluções não podem ser negativas nem superar a quantidade vendida
df_vendas["unidades_devolvidas"] = df_vendas["unidades_devolvidas"].clip(lower=0)
df_vendas["unidades_devolvidas"] = np.minimum(
    df_vendas["unidades_devolvidas"], df_vendas["quantidade"]
)

# Validar base de estoque
df_estoque = df_estoque.dropna(
    subset=["mes", "modelo", "dias_observados", "dias_indisponivel"]
)
df_estoque = df_estoque[
    (df_estoque["dias_observados"] > 0) &
    (df_estoque["dias_indisponivel"] >= 0) &
    (df_estoque["dias_indisponivel"] <= df_estoque["dias_observados"])
].copy()

# Variáveis calculadas
df_vendas["preco_com_desconto"] = (
    df_vendas["preco_unitario"] * (1 - df_vendas["desconto_pct"])
)
df_vendas["faturamento_bruto"] = (
    df_vendas["quantidade"] * df_vendas["preco_com_desconto"]
)
df_vendas["custo_total_produtos"] = (
    df_vendas["quantidade"] * df_vendas["custo_unitario"]
)
df_vendas["margem_bruta_estimada"] = (
    df_vendas["faturamento_bruto"] - df_vendas["custo_total_produtos"]
)
df_vendas["mes"] = df_vendas["data"].dt.to_period("M").astype(str)

print("ETL concluído")
print("Duplicatas identificadas e removidas:", duplicatas)
print("Registros antes da limpeza:", len(df_vendas_original))
print("Registros após a limpeza:", len(df_vendas))
print("\nValores ausentes após a limpeza:")
display(df_vendas.isna().sum().to_frame("valores_ausentes"))


### Justificativa das decisões de limpeza (item 2.1)

| Etapa | O que foi feito | Por quê |
|---|---|---|
| Datas | Convertidas para o tipo data | Permite agrupar as vendas por mês |
| Duplicatas | 1 linha repetida removida (pedido P0003) | Evita contar a mesma venda duas vezes |
| `cor` ausente | Preenchida com "Não informado" | Campo descritivo, não interfere nos cálculos |
| `desconto_pct` ausente | Preenchido com 0 | Hipótese conservadora: venda sem desconto (afeta apenas 1 registro) |
| Valores inválidos | Quantidade ≤ 0 removida; desconto limitado entre 0 e 1; devoluções limitadas à quantidade vendida | Regras preventivas (nenhum caso encontrado nesta planilha) |
| Variáveis novas | `preco_com_desconto`, `faturamento_bruto`, `custo_total_produtos`, `margem_bruta_estimada`, `mes` | Base para os KPIs e gráficos |

**Resultado:** 301 registros antes da limpeza e 300 depois. A aba de estoque não precisou de correções.

In [ ]:
# 4. KPI 1 — taxa de ruptura de estoque
dias_observados = df_estoque["dias_observados"].sum()
dias_indisponiveis = df_estoque["dias_indisponivel"].sum()

taxa_ruptura_geral = (
    dias_indisponiveis / dias_observados * 100
    if dias_observados else np.nan
)

ruptura_modelo = (
    df_estoque.groupby("modelo")
    .agg(
        dias_observados=("dias_observados", "sum"),
        dias_indisponivel=("dias_indisponivel", "sum")
    )
)
ruptura_modelo["taxa_ruptura_pct"] = (
    ruptura_modelo["dias_indisponivel"] /
    ruptura_modelo["dias_observados"] * 100
)

# 5. KPI 2 — ticket médio por pedido
valor_pedido = df_vendas.groupby("pedido_id")["faturamento_bruto"].sum()
ticket_medio = valor_pedido.mean()

# 6. KPI 3 — taxa de devolução por linha e modelo
devolucao_linha = (
    df_vendas.groupby("linha")
    .agg(
        unidades_vendidas=("quantidade", "sum"),
        unidades_devolvidas=("unidades_devolvidas", "sum")
    )
)
devolucao_linha["taxa_devolucao_pct"] = (
    devolucao_linha["unidades_devolvidas"] /
    devolucao_linha["unidades_vendidas"] * 100
)

devolucao_modelo = (
    df_vendas.groupby("modelo")
    .agg(
        unidades_vendidas=("quantidade", "sum"),
        unidades_devolvidas=("unidades_devolvidas", "sum")
    )
)
devolucao_modelo["taxa_devolucao_pct"] = (
    devolucao_modelo["unidades_devolvidas"] /
    devolucao_modelo["unidades_vendidas"] * 100
)

print(f"KPI 1 — Taxa geral de ruptura: {taxa_ruptura_geral:.2f}%")
print(f"KPI 2 — Ticket médio por pedido: R$ {ticket_medio:.2f}")
print("\nKPI 1 — Ruptura por modelo:")
display(ruptura_modelo.round(2))
print("\nKPI 3 — Devolução por linha:")
display(devolucao_linha.round(2))
print("\nKPI 3 — Devolução por modelo:")
display(devolucao_modelo.round(2))


### Justificativa dos KPIs e vínculo com o TAP (item 2.1)

Cada KPI mede o progresso de um Objetivo SMART definido no Termo de Abertura do Projeto (TAP):

| KPI | Fórmula | Resultado atual | Por que foi criado | Objetivo SMART do TAP |
|---|---|---|---|---|
| **KPI 1 — Taxa de ruptura** | dias indisponíveis ÷ dias observados | 16,06% | Mede quanto tempo os modelos ficam sem estoque, que é o problema central da StrideZone | **Objetivo 1:** reduzir a ruptura em 20% (de 16,1% para até 12,8%) |
| **KPI 2 — Ticket médio** | média do valor por pedido | R$ 597,19 | Garante que reduzir rupturas não derrube a receita por pedido | **Objetivo 3:** manter o ticket médio em pelo menos R$ 597 |
| **KPI 3 — Taxa de devolução** | unidades devolvidas ÷ unidades vendidas | 8,93% | Mostra quais modelos geram devolução e orienta a compra | **Objetivo 2:** reduzir a devolução de 8,9% para até 7,5% |

In [ ]:
# 6.1 Estimativa de receita perdida por ruptura
# Hipótese: nos dias com estoque, cada modelo vende a um ritmo diário constante;
# nos dias de ruptura, esse ritmo seria mantido se houvesse produto.
resumo_modelo = ruptura_modelo.join(
    df_vendas.groupby("modelo").agg(
        unidades_vendidas=("quantidade", "sum"),
        faturamento=("faturamento_bruto", "sum")
    )
)
dias_com_estoque = resumo_modelo["dias_observados"] - resumo_modelo["dias_indisponivel"]
resumo_modelo["vendas_por_dia"] = resumo_modelo["unidades_vendidas"] / dias_com_estoque
resumo_modelo["unidades_perdidas_est"] = (
    resumo_modelo["vendas_por_dia"] * resumo_modelo["dias_indisponivel"]
)
resumo_modelo["receita_perdida_est"] = resumo_modelo["unidades_perdidas_est"] * (
    resumo_modelo["faturamento"] / resumo_modelo["unidades_vendidas"]
)
resumo_modelo["pct_do_faturamento_total"] = (
    resumo_modelo["faturamento"] / df_vendas["faturamento_bruto"].sum() * 100
)

display(resumo_modelo[[
    "taxa_ruptura_pct", "pct_do_faturamento_total",
    "unidades_perdidas_est", "receita_perdida_est"
]].sort_values("receita_perdida_est", ascending=False).round(2))

receita_perdida_total = resumo_modelo["receita_perdida_est"].sum()
pct_perdido = receita_perdida_total / df_vendas["faturamento_bruto"].sum() * 100
print(f"Receita perdida estimada por ruptura: R$ {receita_perdida_total:,.2f} "
      f"({pct_perdido:.1f}% do faturamento do período)")

> **Atenção:** a receita perdida é uma **estimativa** baseada na hipótese de demanda diária constante. Serve para dimensionar o problema, não como valor exato.

In [ ]:
# 7. Estatística descritiva do valor dos pedidos
estatisticas = pd.Series({
    "Média (R$)": valor_pedido.mean(),
    "Mediana (R$)": valor_pedido.median(),
    "Desvio padrão (R$)": valor_pedido.std(),
    "Assimetria": valor_pedido.skew(),
    "Mínimo (R$)": valor_pedido.min(),
    "Máximo (R$)": valor_pedido.max()
}, name="valor_pedido")

display(estatisticas.to_frame().round(2))

print("Quantidade vendida por linha:")
display(df_vendas.groupby("linha")["quantidade"].sum().sort_values(ascending=False).to_frame())


In [ ]:
# 7.1 Outliers no valor dos pedidos (regra do boxplot: acima de Q3 + 1,5 x IQR)
q1, q3 = valor_pedido.quantile([0.25, 0.75])
limite_superior = q3 + 1.5 * (q3 - q1)
pedidos_outliers = valor_pedido[valor_pedido > limite_superior]

print(f"Limite superior (Q3 + 1,5 x IQR): R$ {limite_superior:,.2f}")
print(f"Pedidos outliers: {len(pedidos_outliers)} de {len(valor_pedido)} "
      f"({len(pedidos_outliers) / len(valor_pedido) * 100:.1f}%)")
print(f"Participação dos outliers no faturamento: "
      f"{pedidos_outliers.sum() / valor_pedido.sum() * 100:.1f}%")

### Interpretação da estatística descritiva e da distribuição (item 2.2)

**1. Estatística descritiva.** A média dos pedidos (R$ 597,19) é bem maior que a mediana (R$ 459,00), e o desvio padrão (R$ 398,66) equivale a cerca de 67% da média. Ou seja, o valor dos pedidos varia muito: há pedidos de 1 par barato e pedidos de vários pares caros.

**2. Distribuição.** A assimetria é de 1,63 (positiva e forte). A maioria dos pedidos tem valor baixo e poucos pedidos muito altos puxam a média para cima. A distribuição **não é normal**, então a **mediana** representa melhor o pedido típico.

**3. Outliers.** Pela regra do boxplot, os pedidos acima do limite superior são poucos: 10 pedidos (3,3% do total, acima de R$ 1.502,25), que respondem por 10,4% do faturamento. Assim, a maior parte do faturamento vem de pedidos comuns, e não de pedidos excepcionais.

In [ ]:
# 8. Correlação: desconto x quantidade vendida
# Pearson mede associação linear; Spearman mede associação monotônica por postos.
if df_vendas["desconto_pct"].nunique() >= 2 and df_vendas["quantidade"].nunique() >= 2:
    pearson_r, pearson_p = pearsonr(
        df_vendas["desconto_pct"], df_vendas["quantidade"]
    )
    spearman_r, spearman_p = spearmanr(
        df_vendas["desconto_pct"], df_vendas["quantidade"]
    )
    print(f"Pearson r = {pearson_r:.3f}; p-valor = {pearson_p:.4f}")
    print(f"Spearman rho = {spearman_r:.3f}; p-valor = {spearman_p:.4f}")
else:
    pearson_r = spearman_r = np.nan
    print("Não há variação suficiente para calcular as correlações.")

print("A correlação não prova que o desconto causa mudança nas vendas.")


In [ ]:
# 8.1 Quantidade média por faixa de desconto
qtd_por_desconto = df_vendas.groupby("desconto_pct")["quantidade"].agg(["mean", "count"])
qtd_por_desconto.columns = ["quantidade_media", "registros"]
display(qtd_por_desconto.round(2))

### Interpretação da correlação (item 2.2)

Pearson r = 0,278 e Spearman ρ = 0,259, ambos com p-valor menor que 0,001. Trata-se de uma correlação **positiva e fraca, mas estatisticamente significativa**: descontos maiores acompanham pedidos com mais pares (em média 1,39 par sem desconto contra 2,26 pares com 20%). Como r² ≈ 0,077, o desconto explica apenas cerca de 8% da variação da quantidade vendida. **Correlação não prova causalidade.**

In [ ]:
# 9. Gráficos
# Gráfico 1: taxa de ruptura por modelo
ruptura_modelo["taxa_ruptura_pct"].sort_values().plot(kind="bar", figsize=(10, 5))
plt.title("Taxa de ruptura de estoque por modelo (dados fictícios)")
plt.xlabel("Modelo")
plt.ylabel("Taxa de ruptura (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

# Gráfico 2: faturamento mensal
faturamento_mensal = df_vendas.groupby("mes")["faturamento_bruto"].sum()
faturamento_mensal.plot(marker="o", figsize=(10, 5))
plt.title("Faturamento mensal estimado (dados fictícios)")
plt.xlabel("Mês")
plt.ylabel("Faturamento (R$)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Gráfico 3: desconto x quantidade
plt.figure(figsize=(8, 5))
plt.scatter(df_vendas["desconto_pct"] * 100, df_vendas["quantidade"], alpha=0.5)
plt.title("Desconto x quantidade vendida (dados fictícios)")
plt.xlabel("Desconto (%)")
plt.ylabel("Quantidade por registro")
plt.tight_layout()
plt.show()

# Gráfico 4: boxplot dos valores por pedido
plt.figure(figsize=(8, 5))
plt.boxplot(valor_pedido, vert=True)
plt.title("Distribuição dos valores dos pedidos (dados fictícios)")
plt.ylabel("Valor do pedido (R$)")
plt.tight_layout()
plt.show()


In [ ]:
# 9.1 Gráficos complementares
# Gráfico 5: histograma do valor dos pedidos
plt.figure(figsize=(8, 5))
plt.hist(valor_pedido, bins=15, edgecolor="black")
plt.axvline(valor_pedido.mean(), color="black", linestyle="--",
            label=f"Média (R$ {valor_pedido.mean():,.0f})")
plt.axvline(valor_pedido.median(), color="red", linestyle=":",
            label=f"Mediana (R$ {valor_pedido.median():,.0f})")
plt.title("Distribuição do valor dos pedidos (dados fictícios)")
plt.xlabel("Valor do pedido (R$)")
plt.ylabel("Número de pedidos")
plt.legend()
plt.tight_layout()
plt.show()

# Gráfico 6: quantidade média por faixa de desconto
plt.figure(figsize=(8, 5))
(qtd_por_desconto["quantidade_media"]).plot(kind="bar", edgecolor="black")
plt.title("Quantidade média por faixa de desconto (dados fictícios)")
plt.xlabel("Desconto")
plt.ylabel("Pares por pedido (média)")
plt.xticks(range(len(qtd_por_desconto)),
           [f"{d * 100:.0f}%" for d in qtd_por_desconto.index], rotation=0)
plt.tight_layout()
plt.show()

# Gráfico 7: mapa de calor das correlações
colunas_corr = ["desconto_pct", "quantidade", "preco_unitario",
                "unidades_devolvidas", "faturamento_bruto"]
matriz = df_vendas[colunas_corr].corr()
plt.figure(figsize=(8, 6))
plt.imshow(matriz, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlação de Pearson")
plt.xticks(range(len(colunas_corr)), colunas_corr, rotation=35, ha="right")
plt.yticks(range(len(colunas_corr)), colunas_corr)
for i in range(len(colunas_corr)):
    for j in range(len(colunas_corr)):
        plt.text(j, i, f"{matriz.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.title("Mapa de calor das correlações (dados fictícios)")
plt.tight_layout()
plt.show()

### Gráficos escolhidos e por quê (item 2.3)

| Gráfico | Tipo | O que mostra |
|---|---|---|
| 1 | Barras | Ruptura por modelo: quais modelos ficam mais tempo sem estoque |
| 2 | Linha | Evolução do faturamento mês a mês |
| 3 | Dispersão | Relação entre desconto e quantidade vendida |
| 4 | Boxplot | Mediana, dispersão e outliers do valor dos pedidos |
| 5 | Histograma | Formato (assimetria) da distribuição dos valores |
| 6 | Barras | Quantidade média de pares por faixa de desconto |
| 7 | Mapa de calor | Quais variáveis estão mais ligadas entre si |

### Resumo dos principais achados (base para a Análise Crítica — item 2.4)

- **Maior gargalo:** ruptura de estoque. A taxa geral é de 16,06%, mas o **StrideRun X1** (modelo mais vendido) fica indisponível em 25,6% dos dias e o **Urban Casual Lite** em 23,1%.
- **Impacto estimado:** a receita perdida por ruptura (célula 6.1) é de cerca de R$ 39.368, aproximadamente 22% do faturamento do período (só o X1 responde por R$ 22.587).
- **Desconto:** correlação fraca com a quantidade (r = 0,28), então promoção não resolve a falta de estoque.
- **Outliers:** 10 pedidos muito altos (3,3% do total), que respondem por 10,4% do faturamento.
- **Devolução:** taxa geral de 8,93%; o StrideRun X1 lidera com 10,76%.

In [ ]:
# 10. Exportar resultados para CSV e baixar
df_vendas.to_csv("stridezone_vendas_tratadas.csv", index=False, encoding="utf-8-sig")
ruptura_modelo.to_csv("stridezone_kpi_ruptura.csv", encoding="utf-8-sig")
devolucao_linha.to_csv("stridezone_kpi_devolucoes_por_linha.csv", encoding="utf-8-sig")
devolucao_modelo.to_csv("stridezone_kpi_devolucoes_por_modelo.csv", encoding="utf-8-sig")
estatisticas.to_csv("stridezone_estatisticas_ticket.csv", encoding="utf-8-sig")

print("Arquivos CSV exportados. Baixe os que precisar usando as linhas abaixo.")
files.download("stridezone_vendas_tratadas.csv")
files.download("stridezone_kpi_ruptura.csv")
files.download("stridezone_kpi_devolucoes_por_linha.csv")
files.download("stridezone_kpi_devolucoes_por_modelo.csv")
files.download("stridezone_estatisticas_ticket.csv")
